# FIT DNU CONQUER - Phân tích Dữ liệu Chuỗi Thời gian (Time Series)

## Lab 3: Dự báo Ô nhiễm không khí với SARIMA (Seasonal ARIMA)

**Nhóm:** FIT DNU CONQUER

**Chủ đề 5.3.2:** SARIMA – thêm mùa vụ (seasonality)

**Nội dung thực hiện:**

1.  **Q1: Kiểm tra tính mùa vụ (Seasonality Check):**
    * Sử dụng biểu đồ tự tương quan (ACF) để chứng minh chuỗi dữ liệu có tính mùa vụ (đỉnh ở lag 24, 48...).
2.  **Q2: Xây dựng mô hình SARIMA:**
    * Thiết lập tham số mùa vụ $s=24$ (chu kỳ ngày).
    * Thử nghiệm cấu hình $(p,d,q) \times (P,D,Q,s)$.
3.  **Q3: Đánh giá & So sánh:**
    * So sánh hiệu quả giữa ARIMA (không mùa vụ) và SARIMA (có mùa vụ).
    * Đánh giá qua các chỉ số: AIC, RMSE, MAE.
    * Trực quan hóa kết quả dự báo (Forecast vs Actual).

---
# PHẦN 1: CẤU HÌNH VÀ IMPORT THƯ VIỆN

In [7]:
"""
CELL 1: IMPORT LIBRARY & CONFIG
Mục đích: Cấu hình môi trường và import các thư viện cần thiết.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from pathlib import Path

# Thư viện Time Series
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

# Cấu hình đường dẫn
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"✅ Project Root: {PROJECT_ROOT}")

✅ Project Root: C:\lab3huyen


# PHẦN 2: Q1 - CHUẨN BỊ DỮ LIỆU & KIỂM TRA MÙA VỤ

## 2.1. Load dữ liệu và Resample theo giờ
## 2.2. Kiểm tra tính mùa vụ bằng biểu đồ ACF

In [9]:
"""
CELL 2: DATA PREPARATION & EDA FOR SARIMAX
Mục đích: Load dữ liệu, kiểm tra mùa vụ và tương quan thời tiết.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
import os

# 1. Tự động kiểm tra file để tránh FileNotFoundError
possible_paths = [
    PROJECT_ROOT / 'data' / 'processed' / 'beijing_air_quality_cleaned.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'cleaned.parquet',
    Path('C:/lab3huyen/data/processed/beijing_air_quality_cleaned.csv')
]

df = None
for path in possible_paths:
    if path.exists():
        print(f"✅ Đã tìm thấy file tại: {path}")
        df = pd.read_csv(path) if path.suffix == '.csv' else pd.read_parquet(path)
        break

if df is None:
    raise FileNotFoundError("❌ Không tìm thấy dữ liệu! Hãy kiểm tra lại thư mục data/processed")

# 2. Xử lý thời gian và trạm
df['datetime'] = pd.to_datetime(df['datetime'])
station_name = df['station'].unique()[0]
df_station = df[df['station'] == station_name].set_index('datetime').resample('H').mean()

# Nội suy dữ liệu thiếu (Interpolate) [cite: 413, 440]
for col in ['PM2.5', 'TEMP', 'WSPM', 'RAIN']:
    df_station[col] = df_station[col].interpolate(method='linear')

# 3. Vẽ ACF để chứng minh tính mùa vụ (Seasonality) 
plt.figure(figsize=(10, 4))
plot_acf(df_station['PM2.5'].dropna(), lags=48, title=f'ACF PM2.5 - {station_name}')
plt.axvline(x=24, color='red', linestyle='--', label='Chu kỳ 24h')
plt.legend()
plt.show()

# 4. CHỨNG MINH THỜI TIẾT LIÊN QUAN (Yêu cầu Chủ đề 3) 
plt.figure(figsize=(10, 4))
sns.scatterplot(data=df_station.sample(min(1000, len(df_station))), x='WSPM', y='PM2.5', alpha=0.5)
plt.title('Minh chứng: Gió mạnh (WSPM) thì PM2.5 giảm')
plt.show()

FileNotFoundError: ❌ Không tìm thấy dữ liệu! Hãy kiểm tra lại thư mục data/processed